# Eval `v1` on a task sample (4xL4)

Generated by `scripts/make_eval_notebook.py`. Measures per-task **wall time, tool calls, LLM calls, resolved**
for the embedded submission and projects the full scorer run (~120 tasks, 12 h limit).

**Notebook settings (right panel):**
- Accelerator: **GPU L4 x4** · Internet: **off** · Environment: **pin to 2026-07-01** (wheelhouse wheels are cp312)
- Add input: competition **gemma-4-developer-agent**
- Add input: dataset **metric/gemma-4-developer-agent-wheelhouse**
- Add input: model **google/gemma-4 → other → gemma-4-31b-it-qat-w4a16-ct (v2)**

Use **Save Version → Save & Run All (commit)** so it runs in the background, then download `eval_results.zip`
from the Output tab. Resolve rates on public tasks are pessimistic (some gold patches fail in this env);
focus on timing and failure modes.

In [ ]:
SUBMISSION_NAME = 'v1'
TASK_IDS = ["fastapi_14512", "fastapi_14583", "fastapi_14851", "fastapi_14986", "fastapi_15661", "fastapi_15800", "httpx_3672", "requests_6644", "rich_3006", "rich_3469", "rich_3521", "rich_3675"]
SUBMISSION_FILES = {
 "agent.yaml": "name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n",
 "configs/sampling.yaml": "temperature: 1.0\ntop_p: 0.95\nmax_output_tokens: 16384\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n",
 "eval_config.yaml": "# Per-task budgets. The whole run (~120 tasks + sandbox setup + vLLM start)\n# must finish inside 12 h or the submission gets no score.\n# Anchored to the public LB-0.22 notebook (8 min / 28 calls), which is known to\n# finish: in practice the tool-call cap binds first (~9 s per model call), so\n# the average task ends well under the time cap.\nevaluation:\n  timeout_seconds: 180\n  max_tool_calls: 30\n  max_time_minutes: 7\n  max_turns: 70\n",
 "prompts/system.md": "You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue pass. The hidden tests were written from the issue text, so the exact names, messages, types, defaults and behaviours in the issue are the specification.\n\n# HARD LIMITS\n- About 30 tool calls and 7 minutes in total. `get_status` and `submit_patch` are free; everything else counts. When the budget runs out, the working tree is graded as it is. An existing edit counts even without `submit_patch`. An unedited tree always scores zero.\n- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`, `test/`, `testing/`). They are stripped from your patch anyway.\n- Changes to `pyproject.toml`, `setup.cfg`, `tox.ini`, `pytest.ini`, `conftest.py` are stripped from your patch. Never rely on them; never touch `/workspace/pytest.ini` or `/workspace/conftest.py`.\n- Never create files in /workspace except genuinely new source files the issue requires. Scratch and repro scripts go in /tmp (untracked files in /workspace end up in your patch).\n- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.\n- Never look outside /workspace and never pip install. Everything is already installed and the machine is offline.\n\n# CALL PLAN\n- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If unsure, make your best-guess edit in the most likely place now; a wrong edit can be corrected, pure reading cannot score.\n- Calls 13-22: verify and correct.\n- Call 23 onward: no new exploration. Check `git diff`, fix, submit.\n- Always finish with `git diff` followed by `submit_patch`.\n\n# PROCEDURE\n\n## 1. Extract the specification (no tool call)\nFrom the issue, list for yourself: the exact module, function, class, parameter or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code. If the issue is terse (e.g. only a title or \"Fixes #NNNN\"), infer the most likely intended behaviour from the name of the feature and the surrounding code.\n\n## 2. Locate\n- Use exact paths from the issue directly. Otherwise combine independent searches into one run_command: `git grep -n -F '<literal>' -- '*.py' | head -20` for each exact symbol or message, plus `git ls-files | grep -i <keyword> | head -20` for paths.\n- Batch only searches whose inputs are already known. Keep every output short: always pipe through `head`, never dump whole files or the repository tree.\n- Read the relevant implementation plus one relevant caller or existing test. Prefer `sed -n 'A,Bp' file` with explicit ranges (about 40-80 lines, at most three regions per command) or `read_file` with start/end lines.\n- Bug: find where the wrong behaviour is produced (not where it is merely observed); use a caller or test to understand the contract.\n- Feature: find the closest existing sibling (similar function, option, parameter, class) and copy its pattern completely, including registration points: `__all__`, `__init__.py` re-exports, option lists, dispatch tables, type hints, docstrings.\n- Do not repeat a search that already answered the question.\n- If the issue contains a reproducing snippet, you may run it once via `python - <<'EOF' ... EOF` (never save it inside /workspace) to see the real error.\n\n## 3. Edit\n- Use `edit_file` for existing code. Copy `old_string` exactly from what you read, and keep it SHORT: 1-5 unique lines. Never put triple-quoted strings (`\"\"\"` or `'''`) in `old_string` or `new_string`; anchor on a nearby line without them instead. Split large changes into several small edits.\n- If `edit_file` reports no match or multiple matches, re-read the exact region and retry with a smaller unique snippet. Never repeat an identical failed call. After two failures, use one `python - <<'EOF'` script via run_command that asserts the old text occurs exactly once, replaces it, and writes the file.\n- Fix the cause, not the symptom. Do not special-case the example from the issue.\n- Change only what the issue requires. Keep existing signatures, defaults and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.\n- Use the exact exception class and message text from the issue. If the issue only describes the error, copy the wording style of neighbouring error messages.\n- Handle every edge case you listed in step 1.\n- For documentation-example tasks (e.g. FastAPI `docs_src/`), edit the executable example code under `docs_src/`.\n\n## 4. Verify (at most 3 calls)\n- Find the related test file with `git ls-files tests | grep -i <keyword> | head`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k <keyword> 2>&1 | tail -30`.\n- Failures caused by missing fixtures, missing data or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.\n- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one quick `python -c` check that imports the module and exercises the changed behaviour.\n\n## 5. Final check and submit\n- Run `git status --short && git diff`. Confirm: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; no debug prints; no stray files in /workspace (delete them).\n- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited a source file: edit now and submit again.\n- End with a one-sentence summary of the fix.\n\n# DECISION RULES\n- Ambiguous issue: choose the most literal reading of the issue text and act on it. Never ask questions, never stall.\n- Several possible locations: pick the one where the wrong behaviour is produced.\n- Search found nothing: try one different specific string, then edit your best candidate anyway.\n- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.\n- If a tool call is rejected as malformed, simplify its arguments (shorter strings, no triple quotes) rather than resending it unchanged.\n- Never conclude that nothing needs to change. Every task requires a non-empty source patch.\n- Use `get_status` (free) if unsure how much budget is left.\n"
}

In [ ]:
# --- Environment + wheelhouse install (from the official Getting Started notebook) ---
import glob, importlib, logging, os, shutil, subprocess, sys, time, json
from pathlib import Path

print('python', sys.version)
assert sys.version_info[:2] == (3, 12), (
    f'Python {sys.version_info[0]}.{sys.version_info[1]}: the wheelhouse is cp312 only. '
    'Use a copy of the official Getting Started notebook (Copy & Edit keeps its Python 3.12 environment), '
    'then File -> Import Notebook this file into it.')

for k, v in {
    'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1', 'TRANSFORMERS_OFFLINE': '1',
    'VLLM_NO_USAGE_STATS': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
    'NCCL_DEBUG': 'WARN', 'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'LITELLM_TELEMETRY': 'False',
    'LITELLM_LOG': 'WARNING', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true',
}.items():
    os.environ[k] = v
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')
os.environ.setdefault('LOCAL_API_KEY', 'EMPTY')

def configure_logging():
    for name in ('LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm', 'httpx', 'httpcore',
                 'openai', 'urllib3', 'opentelemetry', 'opentelemetry.context'):
        lg = logging.getLogger(name); lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False; lg.handlers.clear()
configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not any(WHEELHOUSE_DIR.glob('swegemma-*.whl')):  # mount layout differs: locate it
    hits = sorted({w.parent for w in Path('/kaggle/input').rglob('swegemma-*.whl')})
    assert hits, 'Wheelhouse dataset not attached (no swegemma-*.whl under /kaggle/input)'
    WHEELHOUSE_DIR = hits[0]
print('WHEELHOUSE_DIR', WHEELHOUSE_DIR)
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    tgt = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not tgt.exists():
        try: os.symlink(w, tgt)
        except OSError: shutil.copyfile(w, tgt)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches(); configure_logging()
print('python', sys.version)

In [ ]:
# --- Write the embedded submission, load the task sample ---
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
if not (DATA_DIR / 'tasks.jsonl').exists():  # locate the competition mount
    hits = sorted(t.parent for t in Path('/kaggle/input').rglob('tasks.jsonl') if (t.parent / 'snapshots').is_dir())
    assert hits, 'Competition data not attached (no tasks.jsonl + snapshots/ under /kaggle/input)'
    DATA_DIR = hits[0]
print('DATA_DIR', DATA_DIR)
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'submission'
shutil.rmtree(AGENT_DIR, ignore_errors=True)
for rel, text in SUBMISSION_FILES.items():
    p = AGENT_DIR / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text, encoding='utf-8')

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
by_id = {t.instance_id: t for t in all_tasks}
tasks = [by_id[i] for i in TASK_IDS if i in by_id]
print(f'{len(tasks)} tasks:', [t.instance_id for t in tasks])

In [ ]:
# --- Start vLLM (from the official Getting Started notebook) ---
import re, litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True

MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': ('--hf-overrides', '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}'),
}
declared = discover_declared_models(AGENT_DIR)
assert len(declared) == 1, declared
MODEL_NAME = next(iter(declared))
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2')
if not (MODEL_PATH / 'config.json').exists():
    # mount layout differs between notebook versions: find a dir named after the model holding config.json
    hits = sorted({c.parent for c in Path('/kaggle/input').rglob('config.json')
                   if MODEL_NAME in str(c.parent)})
    print('model dir candidates:', hits)
    if not hits:
        print('config.json files under /kaggle/input:',
              [str(c.parent) for c in Path('/kaggle/input').rglob('config.json')][:20])
    assert hits, f'Model {MODEL_NAME} not found under /kaggle/input: attach the right variation'
    MODEL_PATH = hits[0]
print('MODEL_PATH', MODEL_PATH)

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
ranks = [r for a in adapter_map.values() if (r := getattr(a, 'rank', None))]
gpu_count = torch.cuda.device_count()
tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

t0 = time.perf_counter()
server_instance = VllmServer(VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
    dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map), max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(ranks) if ranks else 128, scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp, startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(MODEL_NAME, ()),
), adapter_manifest=adapters)
server_instance.start()
VLLM_STARTUP_S = time.perf_counter() - t0
print(f'vLLM up in {VLLM_STARTUP_S:.0f}s (tp={tp}, gpus={gpu_count})')
models = server_instance.create_model_registry(aliases=[MODEL_NAME], model_prefix='openai/',
                                               api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'))

In [ ]:
# --- Phase 1 + Phase 2 on the sample, logging every task ---
import asyncio, concurrent.futures, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(fn, *a, **kw):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(1) as pool:
            return pool.submit(lambda: asyncio.run(fn(*a, **kw))).result()
    return asyncio.run(fn(*a, **kw))

ec_path = next((p for n in ('eval_config.yaml', 'eval_config.yml') if (p := AGENT_DIR / n).is_file()), None)
ev = (yaml.safe_load(ec_path.read_text()) or {}) if ec_path else {}
ev = ev.get('evaluation', ev)
BUDGET = dict(
    timeout_seconds=int(ev.get('timeout_seconds', 300)),
    max_tool_calls=int(ev.get('max_tool_calls', 100)),
    max_time_minutes=float(ev.get('max_time_minutes', 60.0)),
    max_turns=int(ev['max_turns']) if 'max_turns' in ev else (int(ev['max_llm_calls']) if 'max_llm_calls' in ev else None),
)
print('budget', BUDGET)

limits, gen_constraints = build_submission_limits()
RESULTS_DIR = WORKING_DIR / 'results'
shutil.rmtree(RESULTS_DIR, ignore_errors=True); (RESULTS_DIR / 'patches').mkdir(parents=True)
evaluator = Evaluator(EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
    timeout_seconds=BUDGET['timeout_seconds'], verification_timeout_seconds=300,
    max_time_minutes=BUDGET['max_time_minutes'], max_tool_calls=BUDGET['max_tool_calls'],
    max_turns=BUDGET['max_turns'], limits=limits, generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                                                    token_threshold=14336, event_retention_size=5),
    graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels', verbose=False,
))

rows = []
log_path = RESULTS_DIR / 'task_results.jsonl'
for i, task in enumerate(tasks, 1):
    t0 = time.perf_counter()
    try:
        r = run_sync(evaluator.evaluate_task, task=task, task_index=i, total_tasks=len(tasks))
        err = getattr(r, 'error', None) or getattr(r, 'error_message', None)
        patch = (r.agent_patch or '').strip()
        row = dict(id=task.instance_id, repo=task.repo, resolved=bool(r.resolved),
                   exit_code=r.test_exit_code, tool_calls=r.tool_calls,
                   llm_calls=getattr(r, 'total_llm_calls', None),
                   agent_s=round(r.duration_seconds, 1), patch_chars=len(patch), error=err)
        (RESULTS_DIR / 'patches' / f'{task.instance_id}.patch').write_text(patch)
        (RESULTS_DIR / 'patches' / f'{task.instance_id}.test.log').write_text((r.test_output or '')[-20000:])
    except Exception as e:  # keep going; record the failure
        row = dict(id=task.instance_id, repo=task.repo, resolved=False, error=f'{type(e).__name__}: {e}')
    row['wall_s'] = round(time.perf_counter() - t0, 1)
    rows.append(row)
    with log_path.open('a') as fh: fh.write(json.dumps(row) + '\n')
    print(f"[{i}/{len(tasks)}] {row['id']}: resolved={row['resolved']} calls={row.get('tool_calls')} "
          f"llm={row.get('llm_calls')} agent={row.get('agent_s')}s wall={row['wall_s']}s "
          f"patch={row.get('patch_chars')} err={row.get('error')}")

In [ ]:
# --- Summary + 12 h projection ---
import statistics as st, zipfile
ok = [r for r in rows if 'agent_s' in r]
wall = [r['wall_s'] for r in rows]
agent = [r['agent_s'] for r in ok]
summary = dict(
    submission=SUBMISSION_NAME, model=MODEL_NAME, budget=BUDGET, n=len(rows),
    resolved=sum(r['resolved'] for r in rows), empty_patch=sum(r.get('patch_chars', 0) == 0 for r in rows),
    errors=sum(bool(r.get('error')) for r in rows),
    hit_call_cap=sum((r.get('tool_calls') or 0) >= BUDGET['max_tool_calls'] for r in ok),
    hit_time_cap=sum(r['agent_s'] >= BUDGET['max_time_minutes'] * 60 - 5 for r in ok),
    vllm_startup_s=round(VLLM_STARTUP_S),
    agent_s_mean=round(st.mean(agent), 1) if agent else None,
    agent_s_p90=round(sorted(agent)[int(0.9 * (len(agent) - 1))], 1) if agent else None,
    wall_s_mean=round(st.mean(wall), 1),
    tool_calls_mean=round(st.mean([r.get('tool_calls') or 0 for r in ok]), 1) if ok else None,
)
# wall_s includes Phase-2 verification, which the 12 h limit excludes -> projection is conservative
summary['projected_hours_120_tasks'] = round((VLLM_STARTUP_S + 120 * summary['wall_s_mean']) / 3600, 2)
print(json.dumps(summary, indent=1))
(RESULTS_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))
for rel, text in SUBMISSION_FILES.items():
    p = RESULTS_DIR / 'submission' / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text)
with zipfile.ZipFile(WORKING_DIR / 'eval_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(RESULTS_DIR))
print('wrote', WORKING_DIR / 'eval_results.zip')
server_instance.stop()